# 1 - YouTube Views Prediction - Regression

<img src='https://www.heptekno.com/wp-content/uploads/2019/01/youtube-un-challenge-konusundaki-cozumu.jpeg'>


Bu çalışmada YouTube video verilerini kullanarak videoların izlenme sayısını tahmin eden bir regression modeli geliştireceğiz.


## Akış

1. Veriyi yükleme
2. Veriyi inceleme
3. Veri temizleme
4. Feature engineering
5. Kategorik verileri sayısala çevirme
6. Regression modelleri kurma
7. Model sonuçlarını karşılaştırma
8. Sonuç


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Veriyi Yükleme


In [2]:
# Bu projede Kaggle'dan indirilen YouTube datasetini Colab üzerinden zip dosyası olarak açıp kullanacağım.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import zipfile

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/YouTube Analytics Data.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')


In [5]:
import os
os.listdir('/content')[:50]


['.config', 'youtube_recommendation_dataset -.csv', 'drive', 'sample_data']

## 2. Veriyi Okuma ve İnceleme


In [6]:
# Bu bölümde zip içinden çıkan gerçek csv dosyasını okuyup veri setinin yapısını inceleyeceğim.


In [7]:
file_path = '/content/youtube_recommendation_dataset -.csv'

df = pd.read_csv(file_path)
df.head()


,Title,channel_title,published_at,category_id,view_count,like_count,comment_count,favorite_count,duration,definition,caption,engagement_rate,likes_to_views_ratio,comments_to_views_ratio,duration_seconds,video_age_days
0,LA PERVERSA X LA INSUPERABLE X ALOFOKE MUSIC X...,AlofokeMusicSounds,2025-11-16 15:34:55+00:00,10,1405647,140463,9063,0,PT1M51S,hd,False,0.106375,0.099928,0.006448,111,1
1,Moana | Official Teaser,Disney,2025-11-17 17:00:47+00:00,24,2776847,26801,6684,0,PT1M,hd,True,0.012059,0.009652,0.002407,60,0
2,$0 - $1 Trillion Only FISHING in Steal a Brain...,CaylusBlox,2025-11-17 22:57:14+00:00,20,1189857,16174,1827,0,PT18M2S,hd,False,0.015129,0.013593,0.001535,1082,0
3,ALLDAY PROJECT - ‘ONE MORE TIME’ M/V,THEBLACKLABEL,2025-11-17 09:00:07+00:00,10,5319161,0,12869,0,PT3M23S,hd,True,0.002419,0.000000,0.002419,203,1
4,La Lupa | Vendetta Hero Trailer | Overwatch 2,PlayOverwatch,2025-11-17 17:00:06+00:00,20,597542,41742,4728,0,PT3M35S,hd,True,0.077768,0.069856,0.007912,215,0


In [8]:
df.shape


(537, 16)

In [9]:
df.columns


Index(['Title', 'channel_title', 'published_at', 'category_id', 'view_count',
       'like_count', 'comment_count', 'favorite_count', 'duration',
       'definition', 'caption', 'engagement_rate', 'likes_to_views_ratio',
       'comments_to_views_ratio', 'duration_seconds', 'video_age_days'],
      dtype='object')

In [10]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 537 entries, 0 to 536
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Title                    537 non-null    object 
 1   channel_title            537 non-null    object 
 2   published_at             537 non-null    object 
 3   category_id              537 non-null    int64  
 4   view_count               537 non-null    int64  
 5   like_count               537 non-null    int64  
 6   comment_count            537 non-null    int64  
 7   favorite_count           537 non-null    int64  
 8   duration                 537 non-null    object 
 9   definition               537 non-null    object 
 10  caption                  537 non-null    bool   
 11  engagement_rate          537 non-null    float64
 12  likes_to_views_ratio     537 non-null    float64
 13  comments_to_views_ratio  537 non-null    float64
 14  duration_seconds         5

## 3. Veri Temizleme


In [11]:
# Bu bölümde tekrar eden satırları, boş verileri ve veri tiplerini kontrol edeceğim.


In [12]:
df.duplicated().sum()


np.int64(0)

In [13]:
df = df.drop_duplicates()


In [14]:
df.isnull().sum()


,0
Title,0
channel_title,0
published_at,0
category_id,0
view_count,0
like_count,0
comment_count,0
favorite_count,0
duration,0
definition,0


## 4. Feature Engineering


In [15]:
# Bu bölümde model için gereksiz sütunları çıkarıp anlamlı sütunlarla devam edeceğim.


In [16]:
df = df.drop(['Title', 'duration', 'published_at'], axis=1)
df.head()


,channel_title,category_id,view_count,like_count,comment_count,favorite_count,definition,caption,engagement_rate,likes_to_views_ratio,comments_to_views_ratio,duration_seconds,video_age_days
0,AlofokeMusicSounds,10,1405647,140463,9063,0,hd,False,0.106375,0.099928,0.006448,111,1
1,Disney,24,2776847,26801,6684,0,hd,True,0.012059,0.009652,0.002407,60,0
2,CaylusBlox,20,1189857,16174,1827,0,hd,False,0.015129,0.013593,0.001535,1082,0
3,THEBLACKLABEL,10,5319161,0,12869,0,hd,True,0.002419,0.000000,0.002419,203,1
4,PlayOverwatch,20,597542,41742,4728,0,hd,True,0.077768,0.069856,0.007912,215,0


## 5. Kategorik Verileri Sayısala Çevirme


In [17]:
# Model kurmadan önce kategorik sütunları sayısal hale getireceğim.


In [18]:
df = pd.get_dummies(df, drop_first=True)
df.head()


,category_id,view_count,like_count,comment_count,favorite_count,caption,engagement_rate,likes_to_views_ratio,comments_to_views_ratio,duration_seconds,...,channel_title_therainbowgirl,channel_title_touropia,channel_title_videogamedunkey,channel_title_wearecult,channel_title_wittySpace,channel_title_xaviera putri,channel_title_Зуфар Бек,channel_title_ᴅנ тнᴀɴ тuɴ ᴀuɴԍ oғғιcᴀʟ,channel_title_미유 (MIUU AI) 룩북 SPORTS LOOKBOOK AIルックブック,definition_sd
0,10,1405647,140463,9063,0,False,0.106375,0.099928,0.006448,111,...,False,False,False,False,False,False,False,False,False,False
1,24,2776847,26801,6684,0,True,0.012059,0.009652,0.002407,60,...,False,False,False,False,False,False,False,False,False,False
2,20,1189857,16174,1827,0,False,0.015129,0.013593,0.001535,1082,...,False,False,False,False,False,False,False,False,False,False
3,10,5319161,0,12869,0,True,0.002419,0.000000,0.002419,203,...,False,False,False,False,False,False,False,False,False,False
4,20,597542,41742,4728,0,True,0.077768,0.069856,0.007912,215,...,False,False,False,False,False,False,False,False,False,False


## 6. Regression Modelleri


In [19]:
# Bu bölümde birkaç farklı regression modeli kurup sonuçları karşılaştıracağım.


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

x = df.drop('view_count', axis=1)
y = df['view_count']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [21]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append([name, mse, r2])

results_df = pd.DataFrame(results, columns=['Model', 'MSE', 'R2'])
results_df


,Model,MSE,R2
0,Linear Regression,6.671416e+14,0.556690
1,Ridge Regression,9.417711e+14,0.374202
2,Lasso Regression,6.624816e+14,0.559787
3,Random Forest,3.759243e+14,0.750202
4,Gradient Boosting,2.602779e+14,0.827048
5,Extra Trees,4.094856e+14,0.727901


## 7. Sonuçları Karşılaştırma


In [22]:
# Burada hangi modelin daha başarılı olduğuna bakacağım.


In [23]:
results_df.sort_values('R2', ascending=False)


,Model,MSE,R2
4,Gradient Boosting,2.602779e+14,0.827048
3,Random Forest,3.759243e+14,0.750202
5,Extra Trees,4.094856e+14,0.727901
2,Lasso Regression,6.624816e+14,0.559787
0,Linear Regression,6.671416e+14,0.556690
1,Ridge Regression,9.417711e+14,0.374202


## Sonuç


Bu projede en başarılı sonuç Gradient Boosting modeli ile elde edildi. Model 0.827 R2 skoru ile YouTube videolarının izlenme sayısını yüksek oranda açıklayabildi. Random Forest ve Extra Trees modelleri de başarılı sonuç verdi. Lineer modeller ise daha düşük performans gösterdi.
